# Get MMGBSA Energy Terms and Energy Corrections

The energy terms are obtained from gmx_MMGBSA https://valdes-tresanco-ms.github.io/gmx_MMPBSA/dev/
The energy corrections are obtained from custom scripts based on VSGB 2.0 https://doi.org/10.1002/prot.23106 

In [1]:
import pandas as pd
import numpy as np
import os
import tempfile

import MDAnalysis
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis as HBA
from MDAnalysis.coordinates.memory import MemoryReader
from scipy.spatial.distance import euclidean

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, HTML
display(HTML("<style>.container { width:80% !important; }</style>"))

In [2]:
base_dir = "../" #directory containing all trajectories
conversion = 1.7 # Conversion unit (default conversion=1 is kcal/mol. conversion=1.7 converts to kT)

# Load VDW radii mapping from CSV once and create a dictionary
vdw_mapping = pd.read_csv("vdw_mapping.csv")
vdw_dict = dict(zip(vdw_mapping['Atom Name'], vdw_mapping['VDW Radius']))

## Functions to get energy corrections

All corrections except for entropy were based on "The VSGB 2.0 model: A next generation energy model for high resolution protein structure modeling"  https://doi.org/10.1002/prot.23106 

The entropy energy was obtained using the empirical formula found in https://doi.org/10.1093/bioinformatics/btx698

In [3]:
# Get universe from MDAnalysis 
def get_universe(psf, traj):
    '''
    Initializes and returns a Universe object from MDAnalysis. Make sure water is stripped or it will
    take too long to run any analysis.

    Parameters:
    psf (str): Path to the Protein Structure File containing topology information (.psf, .tpr, .prmtop).
    traj (str): Path to the trajectory file containing atomic coordinates (.xtc, .trr, .dcd).

    Returns:
    MDAnalysis.core.universe.Universe: A new MDAnalysis Universe object.
    '''
    # Load the original universe
    u = MDAnalysis.Universe(psf, traj)
    
    return u

### Hydrogen Bonds

In [4]:
# Convert hydrogen bond data into a pandas DataFrame.
def make_hb_df(hbonds):
    '''
    Creates and returns a DataFrame containing detailed information about hydrogen bonds, 
    including frames, donor/acceptor indices, distances, angles, and residue details.

    Parameters:
    hbonds (MDAnalysis.analysis.hbonds.HydrogenBondAnalysis): The HydrogenBondAnalysis object with hydrogen bond results.

    Returns:
    pandas.DataFrame: A DataFrame containing hydrogen bond data with columns for frame, donor/acceptor indices, distances, angles, 
                      and donor/acceptor residue information.
    '''
    # Initialize DataFrame with frame, donor, hydrogen, and acceptor atom indices
    df = pd.DataFrame(hbonds.results.hbonds[:, :4].astype(int),
                      columns=["Frame", "Donor_ix", "Hydrogen_ix", "Acceptor_ix"])

    # Add distance and angle columns from the hydrogen bond results
    df["Distances"] = hbonds.results.hbonds[:, 4]
    df["Angles"] = hbonds.results.hbonds[:, 5]

    # Add donor and acceptor residue names, IDs, and atom names using indices
    df["Donor resname"] = u.atoms[df.Donor_ix].resnames
    df["Acceptor resname"] = u.atoms[df.Acceptor_ix].resnames
    df["Donor resid"] = u.atoms[df.Donor_ix].resids
    df["Acceptor resid"] = u.atoms[df.Acceptor_ix].resids
    df["Donor name"] = u.atoms[df.Donor_ix].names
    df["Acceptor name"] = u.atoms[df.Acceptor_ix].names

    return df


### Extract hydrogen bonds (HBonds) between protein and DNA from a given Universe.
def get_hb(u):
    '''
    Identifies and returns hydrogen bonds between protein and DNA within the specified Universe.

    Parameters:
    u (MDAnalysis.core.universe.Universe): The MDAnalysis Universe object containing protein-DNA simulation data.

    Returns:
    pandas.DataFrame: A DataFrame containing hydrogen bond data with columns for frame, donor/acceptor indices, distances, angles, 
                      and donor/acceptor residue information.
    '''
    
    # Initialize Hydrogen Bond Analysis for interactions between nucleic acids and protein.
    hbonds = HBA(universe=u, between=['nucleic', 'protein'])

    # Run the hydrogen bond analysis
    hbonds.run()
    
    hbonds_df = make_hb_df(hbonds)
    
    return hbonds_df


### Get hydrogen bond strength scaling
def get_aij(row):
    '''
    Determines the strength of hydrogen bond donor (alpha_i) and acceptor (alpha_j) coefficients
    based on the amino acid residue type and atom type.

    Parameters:
    row (pd.Series): A row from the DataFrame containing information on donor and acceptor atoms,
                     including their residue names ('Donor resname', 'Acceptor resname') and atom names.

    Returns:
    tuple: (alpha_i, alpha_j) coefficients for donor and acceptor, respectively.
    '''
    
    # Check if the donor residue is a strong donor:
    # - Strong donors: Nitrogen atoms in charged LYS, ARG, HIS (indicated by 'N' atom name start)
    # - Weak donors: Other donors are considered weak by default
    if row['Donor resname'] in ['LYS', 'ARG', 'HIS'] and row['Donor name'].startswith('N'):
        ai = 1.5
    else:
        ai = 0.5  # Weak donor by default
    
    # Check if the acceptor residue is a strong acceptor:
    # - Strong acceptors: Oxygen atoms in charged ASP, GLU (indicated by 'O' atom name start)
    # - Weak acceptors: Other acceptor residues are considered weak by default
    if row['Acceptor resname'] in ['ASP', 'GLU'] and row['Acceptor name'].startswith('O'):
        aj = -1.5
    else:
        aj = -0.5  # Weak acceptor by default
    
    return ai, aj


### Hydrogen Bonding Energy Correction
def hb_energy(u):
    '''
    Computes the hydrogen bonding energy correction (E_HB) for each hydrogen bond 
    in the DataFrame, based on distances, angles, and donor/acceptor properties.

    Parameters:
    HB_df (pd.DataFrame): DataFrame containing hydrogen bond distances ('Distances') and angles ('Angles'),
                          along with donor and acceptor residue/atom information.

    Returns:
    pd.DataFrame: Updated DataFrame with an additional column ('HB Energies') containing the computed
                  hydrogen bond energy correction values for each bond.
    '''
    HB_df = get_hb(u)
    
    energies = []  # List to store hydrogen bond energy for each row
    r0 = 1.94  # Optimal hydrogen-acceptor distance in Å
    theta0 = np.radians(160)  # Optimal donor-hydrogen-acceptor angle in radians
    
    # Iterate over each hydrogen bond entry in the DataFrame
    for idx, row in HB_df.iterrows():
        ai, aj = get_aij(row)  # Get alpha coefficients for donor and acceptor
        
        # Calculate hydrogen bond energy using the given formula:
        # E_HB = alpha_i * alpha_j * exp[-(rHA - r0)^2] * cos(thetaDHA - theta0)
        # where rHA is the H-A distance, and thetaDHA is the D-H-A angle
        energy = ai * aj * np.exp(- (row['Distances'] - r0)**2) * np.cos(np.radians(row['Angles']) - theta0)
        energies.append(energy)  # Append calculated energy to the list
    
    # Add computed hydrogen bond energies to the DataFrame
    HB_df['HB Energy'] = energies
    HB_df_total = HB_df.groupby('Frame')['HB Energy'].sum().reset_index()
    
    return HB_df_total


### Pi-Pi Stacking

In [5]:
def calculate_centroids(u):
    """
    Calculate the centroids of aromatic planes for specific residues and nucleic acids for each frame in 
    the trajectory as defined in the selections dict.

    Parameters:
    universe (MDAnalysis.Universe): MDAnalysis Universe object containing the structure and trajectory.

    Returns:
    pd.DataFrame: DataFrame with the centroids coordinates for each residue or nucleic acid type.
    Columns are Frame, Residue Type, Residue Number, Centroid X, Centroid Y, Centroid Z 

    """
    
    # Define atom selections for aromatic planes with AMBER-compatible atom labels
    selections = {
        'PHE': "resname PHE and name CG CD1 CD2 CE1 CE2 CZ",
        'TYR': "resname TYR and name CG CD1 CD2 CE1 CE2 CZ",
        'HIS': "resname HIS and name CG ND1 CD2 CE1 NE2",
        'TRP': "resname TRP and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2",
        'ARG': "resname ARG and name CZ NH1 NH2",
        'ASN': "resname ASN and name CG OD1 ND2",
        'GLN': "resname GLN and name CD OE1 NE2",
        'DA': "resname DA and name N1 C2 N3 C4 C5 C6 N7 C8 N9",
        'DG': "resname DG and name N1 C2 N3 C4 C5 C6 N7 C8 N9",
        'DC': "resname DC and name N1 C2 N3 C4 C5 C6",
        'DT': "resname DT and name N1 C2 N3 C4 C5 C6"
    }
    
    # Create a list to store centroid data for each frame, residue, and selection
    centroid_data = []

    # Iterate over each frame in the trajectory
    for ts in u.trajectory:
        for name, selection in selections.items():
            selected_atoms = u.select_atoms(selection)
            if len(selected_atoms) == 0:
                continue  # Skip if no atoms are found for the selection
            
            # Group atoms by unique residues
            for res in selected_atoms.residues:
                res_atoms = res.atoms
                if len(res_atoms) > 0:
                    # Calculate the centroid of the selected atoms for the current residue
                    centroid = np.mean(res_atoms.positions, axis=0)
                    
                    # Append the data for the current frame, residue type, and residue number
                    centroid_data.append({
                        'Frame': ts.frame,
                        'Residue Name': name,
                        'Residue Number': res.resid,
                        'Centroid X': centroid[0],
                        'Centroid Y': centroid[1],
                        'Centroid Z': centroid[2]
                    })

    # Convert the data into a DataFrame
    centroid_df = pd.DataFrame(centroid_data)
    return centroid_df


def calculate_plane_normal(atoms):
    """
    Calculate the normal vector of a plane defined by the positions of atoms.

    Parameters:
    atoms (MDAnalysis.AtomGroup): AtomGroup defining the plane.

    Returns:
    np.array: Normal vector of the plane.
    """
    if len(atoms) < 3:
        raise ValueError("At least three non-collinear atoms are needed to define a plane.")
    
    # Use the first three atoms to define the plane
    v1 = atoms.positions[1] - atoms.positions[0]
    v2 = atoms.positions[2] - atoms.positions[0]
    normal = np.cross(v1, v2)
    return normal / np.linalg.norm(normal)


def project_point_onto_plane(point, plane_point, plane_normal):
    """
    Project a point onto a plane defined by a point and a normal vector.

    Parameters:
    point (np.array): Coordinates of the point to project.
    plane_point (np.array): A point on the plane.
    plane_normal (np.array): Normal vector of the plane.

    Returns:
    np.array: Projected point on the plane.
    """
    vector_to_plane = point - plane_point
    distance_to_plane = np.dot(vector_to_plane, plane_normal)
    projected_point = point - distance_to_plane * plane_normal
    return projected_point


def calculate_distances_and_angles(universe, centroid_df, cutoff=8.0):
    """
    Calculate distances and angles for interactions between protein and DNA residues.

    Parameters:
    universe (MDAnalysis.Universe): MDAnalysis Universe object.
    centroid_df (pd.DataFrame): DataFrame containing centroid coordinates for each residue.
    cutoff (float): Cutoff distance for r1.

    Returns:
    pd.DataFrame: DataFrame with calculated r1, r2, r3, and theta for relevant pairs.
    """
    # Define atom selections for aromatic planes with AMBER-compatible atom labels
    selections = {
        'PHE': "resname PHE and name CG CD1 CD2 CE1 CE2 CZ",
        'TYR': "resname TYR and name CG CD1 CD2 CE1 CE2 CZ",
        'HIS': "resname HIS and name CG ND1 CD2 CE1 NE2",
        'TRP': "resname TRP and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2",
        'ARG': "resname ARG and name CZ NH1 NH2",
        'ASN': "resname ASN and name CG OD1 ND2",
        'GLN': "resname GLN and name CD OE1 NE2",
        'DA': "resname DA and name N1 C2 N3 C4 C5 C6 N7 C8 N9",
        'DG': "resname DG and name N1 C2 N3 C4 C5 C6 N7 C8 N9",
        'DC': "resname DC and name N1 C2 N3 C4 C5 C6",
        'DT': "resname DT and name N1 C2 N3 C4 C5 C6"
    }

    interaction_data = []

    # Group frame data by frame to avoid repeated selections
    frame_groups = centroid_df.groupby('Frame')

    for frame, frame_data in frame_groups:
        # Separate protein and DNA residues
        protein_data = frame_data[frame_data['Residue Name'].isin(['PHE', 'TYR', 'HIS', 'TRP', 'ARG', 'ASN', 'GLN'])]
        dna_data = frame_data[frame_data['Residue Name'].isin(['DA', 'DG', 'DC', 'DT'])]

        # Skip frame if no relevant data is found
        if protein_data.empty or dna_data.empty:
            continue

        # Convert centroid coordinates to arrays for vectorized distance calculation
        protein_centroids = protein_data[['Centroid X', 'Centroid Y', 'Centroid Z']].to_numpy()
        dna_centroids = dna_data[['Centroid X', 'Centroid Y', 'Centroid Z']].to_numpy()

        # Calculate pairwise distances between all protein and DNA centroids
        protein_expanded = protein_centroids[:, np.newaxis, :]  # Shape (n_protein, 1, 3)
        dna_expanded = dna_centroids[np.newaxis, :, :]          # Shape (1, n_dna, 3)
        distance_matrix = np.linalg.norm(protein_expanded - dna_expanded, axis=2)

        # Filter pairs based on cutoff
        protein_indices, dna_indices = np.where(distance_matrix <= cutoff)
        
        for p_idx, d_idx in zip(protein_indices, dna_indices):
            protein_row = protein_data.iloc[p_idx]
            dna_row = dna_data.iloc[d_idx]

            r1_vector = protein_centroids[p_idx] - dna_centroids[d_idx]
            r1 = np.linalg.norm(r1_vector)

            # Get the atoms defining the aromatic planes for calculating r3 and theta
            protein_atoms = universe.select_atoms(
                f"resid {protein_row['Residue Number']} and resname {protein_row['Residue Name']} and ({selections[protein_row['Residue Name']]})"
            )
            dna_atoms = universe.select_atoms(
                f"resid {dna_row['Residue Number']} and resname {dna_row['Residue Name']} and ({selections[dna_row['Residue Name']]})"
            )

            if len(protein_atoms) < 3 or len(dna_atoms) < 3:
                continue  # Skip if there aren't enough atoms to define the plane

            # Calculate plane normals
            protein_normal = calculate_plane_normal(protein_atoms)
            dna_normal = calculate_plane_normal(dna_atoms)

            # Calculate r3 (projection of the protein centroid onto the DNA plane)
            intersection_point = project_point_onto_plane(protein_centroids[p_idx], dna_centroids[d_idx], dna_normal)
            r3_vector = protein_centroids[p_idx] - intersection_point
            r3 = np.linalg.norm(r3_vector)

            # Calculate r2 (horizontal displacement between the planes)
            r2_vector = r1_vector - r3_vector
            r2 = np.linalg.norm(r2_vector)

            # Calculate dihedral angle theta between aromatic planes
            cos_theta = np.abs(np.dot(protein_normal, dna_normal))
            theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))
            theta_degrees = np.degrees(theta)

            # Append results
            interaction_data.append({
                'Frame': frame,
                'Protein Residue': protein_row['Residue Name'],
                'Protein Residue Number': protein_row['Residue Number'],
                'DNA Residue': dna_row['Residue Name'],
                'DNA Residue Number': dna_row['Residue Number'],
                'r1': r1,
                'r2': r2,
                'r3': r3,
                'theta': theta_degrees
            })

    # Convert the results into a DataFrame
    interaction_df = pd.DataFrame(interaction_data)
    return interaction_df

def f(x, x0, x1):
    # Constants for the energy calculation
    A = 2.0
    B = 2.0
    """
    Calculate the f(x) function for the energy formula.

    Parameters:
    x (float): Input value (e.g., r1, r2, r3, or theta).
    x0 (float): Parameter for the function.
    x1 (float): Parameter for the function.

    Returns:
    float: Result of the function.
    """
    return np.exp(-A * np.exp(-B * (x - x0) * (x - x1)))

def calculate_pi_pi_energy(u):
    """
    Calculate the pi-pi stacking energy using the given parameters.

    Parameters:
    r1 (float): Distance between the centers of aromatic planes.
    r2 (float): Horizontal displacement between the aromatic planes.
    r3 (float): Distance from the center in one aromatic plane to the other aromatic plane.
    theta (float): Dihedral angle between the aromatic planes.

    Returns:
    float: Calculated pi-pi stacking energy.
    """
    centroid_df = calculate_centroids(u)
    interaction_df = calculate_distances_and_angles(u, centroid_df)
    
    energies = []
    # Constants for the energy calculation
    C = -3
    
    for idx, rows in interaction_df.iterrows():
        r1 = rows['r1']
        r2 = rows['r2']
        r3 = rows['r3']
        theta = rows['theta']
        
        # Set x0 and x1 based on the given rules
        x0_theta = 0 if theta < 45 else 80
        x1_theta = 20 if theta < 45 else 90
        
        # Calculate the f(x) for each distance/angle
        f_r1 = f(r1, 3.5, 6.5)
        f_r2 = f(r2, 0, 3.5)
        f_r3 = f(r3, 0, 5.0)
        f_theta = f(theta, x0_theta, x1_theta)

        # Calculate the energy
        E = C * f_r1 * f_r2 * f_r3 * f_theta
        energies.append(E)
    
    interaction_df['pi-pi Energy'] = energies
    energy_per_frame = interaction_df.groupby('Frame')['pi-pi Energy'].sum().reset_index()
    energy_per_frame.columns = ['Frame', 'Total pi-pi Energy']
    
    return energy_per_frame


### Hydrophobic term

In [6]:
def get_vdw_radius(atom, vdw_dict):
    """
    Retrieve the VDW radius for a given atom from the VDW mapping dictionary.
    
    Parameters:
    atom (MDAnalysis.core.groups.Atom): Atom object.
    vdw_dict (dict): Dictionary mapping atom names to VDW radii.
    
    Returns:
    float: VDW radius of the atom.
    """
    return vdw_dict.get(atom.name, None)

def calculate_hydrophobic_energy_per_frame(universe, vdw_dict, cutoff=8.0):
    """
    Calculate the hydrophobic energy between protein and DNA heavy atoms for each frame.

    Parameters:
    universe (MDAnalysis.Universe): MDAnalysis Universe object containing the structure and trajectory.
    vdw_dict (dict): Dictionary mapping atom names to VDW radii.
    cutoff (float): Cutoff distance for considering interactions.

    Returns:
    list: Total hydrophobic energy for each frame.
    """
    frame_energies = []

    # Select heavy atoms in the protein and DNA
    protein_heavy_atoms = universe.select_atoms("protein and not name H*")
    dna_heavy_atoms = universe.select_atoms("nucleic and not name H*")
    
    # Get VDW radii for protein and DNA atoms
    protein_vdw_radii = np.array([get_vdw_radius(atom, vdw_dict) for atom in protein_heavy_atoms])
    dna_vdw_radii = np.array([get_vdw_radius(atom, vdw_dict) for atom in dna_heavy_atoms])

    # Remove atoms with missing VDW radii
    valid_protein_mask = ~np.isnan(protein_vdw_radii)
    valid_dna_mask = ~np.isnan(dna_vdw_radii)

    protein_positions = protein_heavy_atoms.positions[valid_protein_mask]
    dna_positions = dna_heavy_atoms.positions[valid_dna_mask]
    protein_vdw_radii = protein_vdw_radii[valid_protein_mask]
    dna_vdw_radii = dna_vdw_radii[valid_dna_mask]

    # Iterate over each frame in the trajectory
    for ts in universe.trajectory:
        frame_hydrophobic_energy = 0.0

        # Update the positions for the current frame
        protein_positions = protein_heavy_atoms.positions[valid_protein_mask]
        dna_positions = dna_heavy_atoms.positions[valid_dna_mask]

        # Calculate all pairwise distances between protein and DNA atoms
        diff_matrix = protein_positions[:, np.newaxis, :] - dna_positions[np.newaxis, :, :]
        distance_matrix = np.linalg.norm(diff_matrix, axis=2)
        
        # Calculate scale matrix
        scale_matrix = 2 * (distance_matrix - protein_vdw_radii[:, np.newaxis] - dna_vdw_radii[np.newaxis, :] - 2) / 3
        
        # Calculate Eij based on scale conditions
        Eij_matrix = np.zeros_like(scale_matrix)
        condition1 = (scale_matrix >= 1)
        condition2 = (-0.1 < scale_matrix) & (scale_matrix < 1)
        condition3 = (scale_matrix <= -1)

        Eij_matrix[condition2] = 0.25 * scale_matrix[condition2]**3 - 0.75 * scale_matrix[condition2] + 0.5
        Eij_matrix[condition3] = 1
        
        # Sum Eij for the current frame and apply the final coefficient
        frame_hydrophobic_energy = np.sum(Eij_matrix) * -0.30
        frame_energies.append(frame_hydrophobic_energy)
    
    frame_energies_df = pd.DataFrame({'Frame':range(len(u.trajectory)), 'Total hydrophobic':frame_energies})

    return frame_energies_df



### Entropy 

## Get energy corrections

In [7]:
# Check if the checkpoint file exists and load it if so
if os.path.exists('energy_corrections_df.csv'):
    all_corrections_df = pd.read_csv('energy_corrections_df.csv')
else:
    all_corrections_df = pd.DataFrame([])

# Loop through each sequence directory
for sequence in os.listdir(base_dir):
    sequence_path = os.path.join(base_dir, sequence)
    if os.path.isdir(sequence_path):  # Check if it's a directory
        # Loop through each simulation directory within the sequence directory
        for run in range(1, 11):
            run_path = os.path.join(sequence_path, str(run))
            if os.path.isdir(run_path):  # Check if it's a directory
                
                # Check if the sequence + run combo already exists in the DataFrame
                if not all_corrections_df.empty and ((all_corrections_df['sequence'] == sequence) & (all_corrections_df['run'] == run)).any():
                    continue
                
                tpr = f'{run_path}/npt_nowat.tpr'
                xtc = f'{run_path}/npt_whole_nowat.xtc'

                # Load the universe
                u = get_universe(tpr, xtc)

                # Calculate HB energies
                hbonds_energies_df = hb_energy(u)

                # Calculate hydrophobic energies
                hydrophobic_energies = calculate_hydrophobic_energy_per_frame(u, vdw_dict)

                # Calculate pi-pi stacking energies
                pi_pi_energies_df = calculate_pi_pi_energy(u)

                # Prepare data for current run
                run_df = pd.DataFrame({
                    'Frame': hbonds_energies_df['Frame'],
                    'HB Energy': hbonds_energies_df['HB Energy'],
                    'Hydrophobic Energy': hydrophobic_energies['Total hydrophobic'],
                    'Pi-Pi Energy': pi_pi_energies_df['Total pi-pi Energy'],
                    'sequence': sequence,
                    'run': run
                })

                # Append the current run data to the main DataFrame
                all_corrections_df = pd.concat([all_corrections_df, run_df], ignore_index=True)

        # Save the updated DataFrame to a CSV file as a checkpoint
        all_corrections_df.to_csv('energy_corrections_df.csv', index=False)  # Save with no index

## Read energy terms from gmx_MMPBSA

In [8]:
# Initialize the main dataframe to store all data
all_mmgbsa_df = pd.DataFrame([])

# Loop through each sequence directory
for sequence in os.listdir(base_dir):
    sequence_path = os.path.join(base_dir, sequence)
    if os.path.isdir(sequence_path):  # Check if it's a directory
        # Loop through each simulation directory within the sequence directory
        for run in os.listdir(sequence_path):
            run_path = os.path.join(sequence_path, run)
            if os.path.isdir(run_path):  # Check if it's a directory
                # Construct the path to the FINAL_RESULTS_strain.csv file
                filename = os.path.join(run_path, 'FINAL_RESULTS_strain.csv')
                
                # Initialize a temporary dataframe for the current file
                mmgbsa_df = pd.DataFrame([])
                
                if os.path.exists(filename):
                    data = open(filename, 'r').readlines()
                    for i in range(len(data)):
                        line = data[i].strip('\n')
                        if 'Terms' in line:
                            energy_system = line
                        elif 'Frame' in line:
                            start_line = i
                        elif line == '' and data[i-1] != '\n':
                            end_line = i - 1
                            # Read the CSV for the specific range and set index
                            df = pd.read_csv(filename, skiprows=start_line, nrows=end_line-start_line)
                            df = df.set_index('Frame #')
                            df.columns = pd.MultiIndex.from_product([[energy_system], df.columns])
                            mmgbsa_df = pd.concat([mmgbsa_df, df], axis=1)
                            mmgbsa_df = mmgbsa_df.multiply(conversion)
                    
                    # Add columns for sequence and run
                    mmgbsa_df['sequence'] = sequence
                    mmgbsa_df['run'] = run
                    
                    # Concatenate the current dataframe with the main dataframe
                    all_mmgbsa_df = pd.concat([all_mmgbsa_df, mmgbsa_df], axis=0)



# Display the final dataframe
all_mmgbsa_df

Complex Energy Terms                                            \
                        BOND         ANGLE         DIHED       VDWAALS   
Frame #                                                                  
1                9414.821204  22802.485815  38485.391027 -23698.916708   
2               10052.587560  23779.932078  38110.966384 -24017.883407   
3                9922.294800  23796.636278  38301.644827 -24010.366517   
4                9941.003504  23111.847599  38264.895587 -23982.052898   
5               10430.436564  23681.126735  38390.344129 -24143.666033   
...                      ...           ...           ...           ...   
37              10004.813548  23172.984971  38237.166615 -23519.764163   
38               9727.690870  23080.109619  38201.670190 -23666.259997   
39               9854.475748  23888.258815  38113.304972 -23745.688468   
40               9544.111712  23251.411190  38497.000446 -23446.349204   
41               9554.969442  23433.904575  38423.501966 -23723.054277   

                                                                            \
                 EEL       1-4 VDW      1-4 EEL           EGB        ESURF   
Frame #                                                                      
1        3228.003129  10712.988107 -5622.967804 -22216.669521  1392.044507   
2        3308.851457  10634.394846 -5633.073845 -22273.046196  1405.992514   
3        3300.248794  10745.227213 -5625.807518 -22305.953470  1404.990262   
4        3494.351598  10697.369680 -5606.096562 -22488.029250  1393.631406   
5        3295.905702  10594.722371 -5694.712343 -22250.161442  1393.380843   
...              ...           ...          ...           ...          ...   
37       1264.006814  10717.080636 -2448.418115 -21865.296674  1424.116571   
38       1394.633658  10733.784836 -2467.627945 -21966.607647  1435.809511   
39       1444.662737  10899.156416 -2513.982100 -21994.420140  1425.703470   
40       1259.830764  10735.872861 -2435.889965 -21890.687058  1412.256589   
41       1309.525759  10825.824978 -2403.901422 -21954.163018  1410.920253   

                       ... Delta Energy Terms                            \
                 GGAS  ...                EEL 1-4 VDW 1-4 EEL       EGB   
Frame #                ...                                                
1        55321.971812  ...          -1886.830     0.0     0.0  1841.253   
2        56235.858594  ...          -1881.424     0.0    -0.0  1835.847   
3        56429.794356  ...          -1895.687     0.0    -0.0  1851.589   
4        55921.234987  ...          -1857.573     0.0     0.0  1814.002   
5        56554.157125  ...          -1897.268    -0.0    -0.0  1847.934   
...               ...  ...                ...     ...     ...       ...   
37       57427.786785  ...          -1887.255    -0.0    -0.0  1840.913   
38       57004.084752  ...          -1881.220    -0.0     0.0  1833.093   
39       57940.188120  ...          -1871.547     0.0    -0.0  1821.023   
40       57405.904283  ...          -1890.740    -0.0     0.0  1839.808   
41       57420.854542  ...          -1859.834     0.0     0.0  1812.387   

                                              \
          ESURF      GGAS     GSOLV    TOTAL   
Frame #                                        
1       -36.482 -2138.940  1804.771 -334.169   
2       -36.023 -2119.917  1799.824 -320.110   
3       -35.802 -2142.408  1815.787 -326.638   
4       -34.799 -2093.244  1779.203 -314.041   
5       -34.391 -2127.482  1813.560 -313.922   
...         ...       ...       ...      ...   
37      -30.226 -2090.082  1810.687 -279.412   
38      -30.515 -2091.221  1802.595 -288.626   
39      -30.702 -2066.503  1790.321 -276.182   
40      -31.671 -2081.837  1808.137 -273.700   
41      -30.855 -2061.454  1781.532 -279.922   

                                            sequence run  
                                                          
Frame #                             

In [9]:
# Remove the multi-index and flatten the column structure
all_mmgbsa_df.columns = [col[1] if col[0] == 'Delta Energy Terms' else col[0] for col in all_mmgbsa_df.columns]

# Ensure 'sequence' and 'run' columns are still present with their correct labels
columns_to_keep = ['sequence','VDWAALS', 'EEL', 'EGB', 'ESURF']

# Select the relevant columns from the DataFrame
all_mmgbsa_df = all_mmgbsa_df[columns_to_keep].reset_index().drop(columns=['Frame #'])

In [10]:
# Ensure 'sequence' and 'run' columns are still present with their correct labels
columns_to_keep = ['sequence','VDWAALS', 'EEL', 'EGB', 'ESURF']

# Select the relevant columns from the DataFrame
all_mmgbsa_df = all_mmgbsa_df[columns_to_keep].reset_index().drop(columns=['Frame #'])
all_mmgbsa_df[columns_to_keep][all_mmgbsa_df['sequence'] != 'MycMax_PDB']

KeyError: "['Frame #'] not found in axis"

In [ ]:
all_mmgbsa_df['sequence'] = all_mmgbsa_df['sequence'].apply(lambda x: x.split('_')[1])
all_mmgbsa_df[columns_to_keep][all_mmgbsa_df['sequence'] != 'PDB'].to_csv('rawdat.csv',index=None)